In [501]:
import pickle # Load refs and annotations
import json
import os
import pandas as pd
import numpy as np
import pprint
import json
import cv2
import random

from typing import Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchvision.transforms as transforms
from torchvision.utils import draw_bounding_boxes
from torchvision import models
import torchmetrics

import pytorch_lightning as pl
from pytorch_lightning.utilities.types import STEP_OUTPUT

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import CLIPProcessor, CLIPModel

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import clip
from ultralytics import YOLO
from PIL import Image, ImageDraw

from ipywidgets import FloatProgress
import math 
from torch.nn.modules.batchnorm import _BatchNorm

In [502]:
device = torch.device('cpu')

In [503]:
clip_model, clip_preprocess = clip.load("RN50", device=device)

In [504]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_area = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1) # +1 to avoid max(0,0) therefore avoiding 

    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / union_area

In [505]:
class MetricMeter:
    def __init__(self, name="Default", threshold = 0.5, log_dir='./logs/default_run'):
        self.name = name
        self.metrics = []
        self.epochs = []
        self.threshold = threshold
        self.reset()
        self.best_cases_iou = []  # To store the best cases
        self.worst_cases_iou = []
        self.best_cases_sim = []  # To store the best cases
        self.worst_cases_sim = []
        self.writer = SummaryWriter(log_dir=log_dir)

# TODO add SummaryWriter to plot

    def reset(self):
        self.count = 0
        self.iou = 0
        self.epoch = 0
        self.correct_bboxes, self.overall = 0,0
        self.semantic = 0
        self.metrics = []
    
    def add_best_iou(self, item):
        if len(self.best_cases_iou) < 5:
            self.best_cases_iou.append(item)
        else:
            min_best_case = min(self.best_cases_iou, key=lambda x: (x['iou']))
            if item['iou'] > min_best_case['iou']:
                self.best_cases_iou.remove(min_best_case)
                self.best_cases_iou.append(item)
    
    def add_worst_iou(self, item):
        if len(self.worst_cases_iou) < 5:
            self.worst_cases_iou.append(item)
        else:
            max_worst_case = max(self.worst_cases_iou, key=lambda x: (x['iou']))
            if (item['iou'] < max_worst_case['iou']):
                self.worst_cases_iou.remove(max_worst_case)
                self.worst_cases_iou.append(item)
    
    def add_best_confidence(self, item):
        if len(self.best_cases_sim) < 5:
            self.best_cases_sim.append(item)
        else:
            min_best_case = min(self.best_cases_sim, key=lambda x: (x['confidence']))
            if item['confidence'] > min_best_case['confidence']:
                self.best_cases_sim.remove(min_best_case)
                self.best_cases_sim.append(item)
    
    def add_worst_confidence(self, item):
        if len(self.worst_cases_sim) < 5:
            self.worst_cases_sim.append(item)
        else:
            max_worst_case = max(self.worst_cases_sim, key=lambda x: (x['confidence']))
            if (item['confidence'] < max_worst_case['confidence']):
                self.worst_cases_sim.remove(max_worst_case)
                self.worst_cases_sim.append(item)


    def update(self, iou, confidence,filename,bbox_e,bbox_gt):
        self.count += 1
        self.iou += iou
        if(iou >= self.threshold):
            self.correct_bboxes += 1
        self.semantic += confidence

        iteration = {
            'run_loc_acc': self.iou / self.count,
            'run_gro_acc': self.correct_bboxes / self.count,
            'run_sem_acc': self.semantic / self.count,
        }

        item = {
            'iou': iou,
            'confidence':confidence,
            'ground_truth': bbox_gt,
            'candidate': bbox_e,
            'path':filename
        }
        self.metrics.append([iteration])
        self.add_best_confidence(item)
        self.add_worst_confidence(item)
        self.add_best_iou(item)
        self.add_worst_iou(item)
        self.writer.add_scalar('Localization Loss', self.iou / self.count, self.count)
        self.writer.add_scalar('Grounding Accuracy',  self.correct_bboxes / self.count, self.count)
        self.writer.add_scalar('Semantic Similarity', self.semantic / self.count, self.count)

    def new_epoch(self):
        self.epoch += 1
        self.writer.add_scalar('Localization Loss', self.iou / self.count, self.epoch)
        self.writer.add_scalar('Grounding Accuracy',  self.correct_bboxes / self.count, self.epoch)
        self.writer.add_scalar('Semantic Similarity', self.semantic / self.count, self.epoch)
        self.epochs.append(self.metrics,self.best_cases_iou,self.best_cases_sim,self.worst_cases_iou,self.worst_cases_sim)
        self.reset()

    def __repr__(self):
        text = f"{self.name}: {self.avg:.8f}"
        return text
    
    def print_iteration(self):
        print(f"Localization accuracy = {self.iou / self.count}, Grounding Accuracy = {self.correct_bboxes / self.count}, Semantic Similarity = {self.semantic / self.count}")

    def get_best_iou_cases(self):
        return sorted(self.best_cases_iou, key=lambda x: x['iou'], reverse=True)

    def get_worst_iou_cases(self):
        return sorted(self.worst_cases_iou, key=lambda x: x['iou'])

    def get_best_pred_cases(self):
        return sorted(self.best_cases_sim, key=lambda x: x['confidence'], reverse=True)

    def get_worst_pred_cases(self):
        return sorted(self.worst_cases_sim, key=lambda x: x['confidence'])
    
def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group["lr"]

In [506]:
with open("./refcocog/annotations/refs(umd).p", "rb") as fp:
  refs = pickle.load(fp)

# 'annotations' will be a dict object mapping the 'annotation_id' to the 'bbox' to make search faster
with open("./refcocog/annotations/instances.json", "rb") as fp:
  data = json.load(fp)
  annotations = dict(sorted({ann["id"]: ann["bbox"] for ann in data["annotations"]}.items()))

In [507]:
def getcaption(elem):
    li = []
    for e in elem["sentences"]:
        li.append(e['raw'])
    return li

In [508]:
annotations.keys

<function dict.keys>

In [527]:
from torchvision.ops import box_convert
class RefCOCOG(Dataset):
    """
    Args:
        The dataset will be the raw data wothput any tipe of preprocessing
        {
            'file_name': 
            'caption':
            'ann_id': needed to extract the relative bbox from the .json file
            'bbox': values are set like following:
                - x 
                - y
                - width 
                - height
        }
    """
    def __init__(self, refs, model, preprocess, annotations, split="train", device = 'cpu', count = 160):
        
        self.clip_model, self.clip_preprocess = model, preprocess
        self.device = device
        self.images = []
        self.texts = []
        self.filepaths =[]
        self.gt = []
        self.cls = []
        temp = 0
        for elem in [d for d in refs if d["split"]==split]:
            
            # Retrieve Single image
            file_name = os.path.join("./refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg')
            image = Image.open(file_name)

            # Retrieve possible ground truth measures
            cls = elem['category_id']
            gt = annotations[elem['ann_id']]
            bbox_tnsor = torch.tensor(gt, device=self.device)
            new_bbox = box_convert(bbox_tnsor, 'xywh', 'xyxy')
            print(gt)
            # Get all texts related to the picture
            sentences = elem['sentences']
            for i in sentences:
                self.texts.append(clip.tokenize(i['raw']))
                self.images.append(self.clip_preprocess(image))
                self.gt.append(new_bbox)
                self.cls.append(cls)
                self.filepaths.append(file_name)
            temp += 1
            if (temp > count):
                break

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        images = self.images[idx]
        gt = self.gt[idx]
        cls = self.cls[idx]
        filename = self.filepaths[idx]
        return text, images, gt, cls, filename

    def __call__(self, idx):
        print(json.dumps(self.dataset[idx], indent=4))


In [528]:
# create dataset and dataloader
dataset_test = RefCOCOG(refs, clip_model, clip_preprocess, annotations, split="test")
dataset_valid = RefCOCOG(refs, clip_model, clip_preprocess, annotations, split="train")
dataset_eval = RefCOCOG(refs, clip_model, clip_preprocess, annotations, split="eval")
#print(dataset_valid[1]['file_name'])
print("---------------------------------------------------")
#plt.imshow(Image.open(dataset[2]["file_name"]))
# dataloader_test = DataLoader(dataset_test, batch_size=16, collate_fn=collate_fn, num_workers=0)
# dataloader_valid = DataLoader(dataset_test, batch_size=16, collate_fn=collate_fn, num_workers=0)
# dataloader_eval = DataLoader(dataset_eval, batch_size=16, collate_fn=collate_fn, num_workers=0)
dataloader_test = DataLoader(dataset_test, batch_size=16)
dataloader_valid = DataLoader(dataset_test, batch_size=16)
dataloader_eval = DataLoader(dataset_eval, batch_size=16)


[374.31, 65.06, 136.04, 201.94]
[93.95, 83.29, 504.61, 290.57]
[338.8, 82.19, 147.34, 157.37]
[45.2, 166.76, 147.45, 179.73]
[496.24, 82.81, 82.8, 168.71]
[375.98, 196.78, 61.71, 178.22]
[39.28, 157.15, 255.05, 196.25]
[182.85, 191.93, 100.03, 155.69]
[40.36, 209.01, 188.83, 429.55]
[305.65, 213.04, 333.63, 198.03]
[392.41, 187.79, 184.67, 213.14]
[56.13, 169.4, 582.37, 256.6]
[349.25, 251.88, 150.75, 116.68]
[325.54, 311.24, 154.46, 159.08]
[227.73, 80.81, 370.99, 341.6]
[62.28, 142.57, 205.15, 337.43]
[328.15, 324.19, 182.43, 74.75]
[353.19, 182.13, 141.78, 188.76]
[159.87, 268.35, 244.1, 60.83]
[2.88, 124.74, 635.22, 297.46]
[223.56, 240.49, 150.63, 101.85]
[137.29, 269.85, 171.06, 166.52]
[263.92, 258.96, 176.45, 91.72]
[118.21, 66.1, 194.91, 113.01]
[256.16, 401.06, 139.29, 116.22]
[194.87, 215.48, 211.0, 202.63]
[144.26, 140.35, 128.98, 288.65]
[198.33, 216.05, 185.17, 191.11]
[303.75, 72.88, 246.76, 156.29]
[320.02, 117.7, 250.53, 248.7]
[173.54, 45.12, 303.73, 141.38]
[81.18, 1

In [529]:
# def pad_image(image):
#     """
#     Performs bottom-right padding of the original image to 640x640 (max size of images in the dataset).
#     Bottom-right padding prevents corruption of bounding boxes.

#     ### Arguments
#     image: a PIL.Image to transform
#     """
#     padded_width, padded_height = 640, 640
#     original_height, original_width = image.shape[:2]
#     bottom_padding = padded_height - original_height
#     right_padding = padded_width - original_width
#     top_padding = 0
#     left_padding = 0
    
#     padded_image = cv2.copyMakeBorder(image, top_padding, bottom_padding, left_padding, right_padding, cv2.BORDER_CONSTANT, value=[0, 0, 0])

#     return padded_image 

# def collate_fn(batch):
#     images = []
#     data = {}

#     #Stores all images in a list
#     for sample in batch:
#         text = sample[0]
#         images = sample[0]
#         gt = sample[0]
#         text = sample[0]
#         image = cv2.imread(sample["file_name"], 3)
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         image = pad_image(image=image)

#         y_ = image.shape[0]
#         x_ = image.shape[1]


#         images.append(transform(image))

#         data['raw'] = sample['raw']
#         x,y,z,c = sample['bbox'][0:4]
#         y1=y 
#         x1=x 
#         y2=(y + c)
#         x2=(x + z)

#         data['bbox'] = [x,y,x2,y2]
#         data['filename'] = sample["file_name"]
            
#     images = torch.stack(images, dim=0)
#     """
#     for key in batch[0].keys():
#         #if key != "file_name":
#         #    data[key] = [sample[key] for sample in batch]
#         data[key] = [sample[key] for sample in batch]
#         if( key == 'bbox'):
#             x,y,z,c = sample[key][0:4]
#             y1=y 
#             x1=x 
#             y2=(y + z)
#             x2=(x + c)
#     """
#     return images, data

# transform = transforms.Compose([
#     transforms.ToTensor(),
# ])

In [576]:
##########################################################
# YolottoClip - Just for Zero-Shotting the dataset
##########################################################
MOMENTUM = 0.1 #Default should be 3e-4
class ConvBNReLU(nn.Module):
    '''Module for the Conv-BN-ReLU tuple.'''

    def __init__(self, c_in, c_out, kernel_size, stride, padding, dilation):
        super(ConvBNReLU, self).__init__()
        self.conv = nn.Conv2d(
                c_in, c_out, kernel_size=kernel_size, stride=stride, 
                padding=padding, dilation=dilation, bias=False)
        self.bn = nn.SyncBatchNorm(c_out, momentum=MOMENTUM)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class External_attention(nn.Module):

    '''
    Arguments:
        c (int): The input and output channel number.
    '''
    def __init__(self, c):
        super(External_attention, self).__init__()
        
        self.conv1 = nn.Conv2d(c, c, 1) #Convolution to linear layer
        self.al4 = 0
        self.k = 64
        self.fc0 = ConvBNReLU(2048, 512, 3, 1, 1, 1)
        self.linear_0 = nn.Conv1d(c, self.k, 1, bias=False)
        self.norm_layer = nn.SyncBatchNorm(c, momentum=MOMENTUM)
        self.linear_1 = nn.Conv1d(self.k, c, 1, bias=False)
        self.linear_1.weight.data = self.linear_0.weight.data.permute(1, 0, 2)        
        self.fc1 = nn.Sequential(
            ConvBNReLU(512, 256, 3, 1, 1, 1),
            nn.Dropout2d(p=0.1))
        self.conv2 = nn.Sequential(
            nn.Conv2d(c, c, 1, bias=False),
            self.norm_layer)    
        self.fc2 = nn.Conv2d(256, 80, 1)   
        self.final_linear = nn.Sequential(
            nn.Linear(3920, 1024)
        )
        
        for m in self.modules():
            if isinstance(m, nn.Conv2d): # Kaiming Initialiaztion
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.Conv1d):# He Initialization
                n = m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, _BatchNorm): # BatchNorm Initialization all setted to 1
                m.weight.data.fill_(1)
                if m.bias is not None:
                    m.bias.data.zero_()

    def process_features(self, tensor, shift=1):
        cycled_tensor = torch.roll(tensor, shifts=shift, dims=0)
        outputs = []
        
        for i in range(cycled_tensor.shape[0]):
            # Extract each slice along the first dimension
            tensor_slice = cycled_tensor[i]
            
            # Pass the slice through the model's forward method
            output = self.forward(tensor_slice.unsqueeze(0))  # Unsqueeze to add the batch dimension back
            
            # Store the output
            outputs.append(output)
        
        # Combine the outputs back into a single tensor if needed
        combined_output = torch.cat(outputs, dim=0)
        
        return combined_output


    def forward(self, x):
        # print(x.shape)
        x = self.fc0(x)
        idn = x
        x = self.conv1(x)

        b, c, h, w = x.size()
        n = h*w
        x = x.view(b, c, n)   # b * c * n 

        attn = self.linear_0(x) # b, k, n
        attn = F.softmax(attn, dim=-1) # b, k, n

        attn = attn / (1e-9 + attn.sum(dim=1, keepdim=True)) #  # b, k, n
        x = self.linear_1(attn) # b, c, n

        x = x.view(b, c, h, w)
        x = self.conv2(x)
        x = x + idn
        x = F.relu(x)
        x = self.fc1(x)
        x = self.fc2(x)
        x = x.view(1, -1)
        # print(x.shape)
        x = self.final_linear(x)

        return x
    
class ExternalYoloClip(nn.Module):
    def __init__(self, clip_model, clip_preprocess, device = 'cpu'):
        super(ExternalYoloClip, self).__init__()
        self.yolo = YOLO("yolov8n.pt")
        self.EA = External_attention(512)
        self.clip_model, self.clip_preprocess = clip_model, clip_preprocess
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.processing = []
        self.similarities = []
        self.device = device

    def infer_bboxes(self, image_path):
        results = self.yolo(image_path, verbose=False)
        # print(results[0])
        bboxes = results[0].boxes.xyxy
        cls = results[0].boxes.cls
        return bboxes,cls

    def encode_image(self, image):
        # Encode the image using the CLIP model
        with torch.no_grad():
            image_features = self.clip_model.encode_image(image)

        return image_features
    
    def preprocess_images(self, cropped_images):
        # preprocess with CLIP each cropped PIL image(converts each image in a image
        # of size [3,224,224])
        processing = []
        for image in cropped_images:
            processed_img = self.clip_preprocess(image).to(self.device)
            # proc = self.clip_model.encode_image(processed_img)
            processing.append(processed_img)
        preprocessed = torch.tensor(np.stack(processing))
        # preprocessed = torch.stack(processing).to(self.device) # return a single tensor
        return preprocessed


    def encode_text(self, text):
        with torch.no_grad():
            text_features = self.clip_model.encode_text(text)
        return text_features
    
    def hook_fn(self, module, input, output):
        self.al = output

    def forward(self, images, text):
        # print(images.shape)
        #bl = 0 # input of layer4 of CLIP's ResNet
        hook_handle = clip_model.visual.layer4.register_forward_hook(self.hook_fn) # handle to retrieve output
        image_features = self.encode_image(images)
        # print(image_features)
        # print(70*'-')
        # # print(self.al.shape)
        image_features = self.EA.process_features(self.al)
        # print(image_features)
        # print(70*'-')
        # print(image_features.shape)
        # print(70*'-')
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = self.encode_text(text)
        # temp = []
        # temp.append(text_features)
        # temp.append(text_features)
        # temp.append(text_features)
        # text_features2 = torch.cat(temp, dim=0)
        # print(text_features)
        # print(70*'-')
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        # text_features2 = text_features2 / text_features2.norm(dim=-1, keepdim=True)
        logit_scale = self.logit_scale.exp()
        # normalized featuresim
        similarity = (image_features @ text_features.t())
        # print(similarity.cpu())
        # similarity = (image_features @ text_features2.t())
        # print(similarity.cpu())
        scaled_similarity = logit_scale * similarity
        logits_per_image = scaled_similarity
        logits_per_text = scaled_similarity.t()

        # print(text_features.shape)
        # # cosine similarity as logits
        # logits_per_image = logit_scale * (image_features @ text_features.t())
        # print("Logits = " + str(logits_per_image))
        # logits_per_text = logits_per_image.t()

        # shape = [global_batch_size, global_batch_size]
        # hook_handle.remove()

        return logits_per_image, logits_per_text
    
    def calculate_best_bbox(self, images, caption, filenames):
        images = images.to(device)
        print(len(images))
        ex_bbox,clss = self.infer_bboxes(images[0])

        print(ex_bbox)
        for bbox in ex_bbox:
            temp = cv2.imread(image_path)
            image = np.zeros((temp.shape[0], temp.shape[1], temp.shape[2]), dtype=np.uint8)
            image[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])] = temp[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])]
            image = Image.fromarray(image)
            images.append(image)
            #image = self.clip_preprocess(image).unsqueeze(0).to(device)

        preprocessed = self.preprocess_images(images)

        similarities, _, _ = self.forward(preprocessed, caption)

        def logit_to_prob_multi_class(logits):
            return torch.sigmoid(logits)
        
        sim = logit_to_prob_multi_class(similarities)
        print(len(similarities))
        print(len(ex_bbox))
        print(sim)

        max = "lol"

        # candidates.append((matching_score,bbox))

        # if matching_score > best_score:
        #     best_score = matching_score
        #     best_bbox = bbox
        #     logit_text = logits_per_text
        #     logit_img = logits_per_image

        return similarities
    

In [577]:
NUM_EPOCHS = 10
count = 0
iou_threshold = 0.5
loss_meter = MetricMeter(name = 'TrainingEA', threshold=iou_threshold)

running_loc_acc = 0.0
correct_bboxes = 0
overall = 0
semantic = 0
semantic_similarity = 0
running_ga = 0
sem = 0
loc_acc = 0
model = ExternalYoloClip(clip_model,clip_preprocess,device='cpu')

In [578]:
loss_meter.reset()

In [579]:
lr = 0.0001
wd = 0.002
alpha = 1 # to decrease lr over time

In [580]:
cost = nn.CrossEntropyLoss()
optimizer = torch.optim.Adadelta(model.parameters(), lr=lr, weight_decay = wd)

In [581]:
cumulative_accuracy = 0.0
cumulative_loss = 0.0
overall = 0

loop = tqdm(dataloader_test, position=0, leave=True)
for _,data in enumerate(loop):
    
    
    texts = data[0]
    texts = texts.squeeze(1).to(device)
    images = data[1]
    gts = data[2]
    clss = data[3]
    filename = data[4]

    optimizer.zero_grad()

    # Build Data for training pass

    # since confidence is directly how much bbox and text "resembles" each other 
    li, lt = model.forward(images,texts)

    # Construct the ground truth
    ground_truth = torch.arange(len(images),dtype=torch.long,device=device)
    img_loss = cost(li, ground_truth)
    desc_loss = cost(lt, ground_truth)
    loss = (img_loss + desc_loss)/2
    loss.backward()
    optimizer.step()

    # Keep track of loss and accuracy metrics to see epochs progress
    overall += 16 #batch_size
    cumulative_loss += loss.item()

    _, predicted = li.max(dim=1)
    cumulative_accuracy += predicted.eq(ground_truth).sum().item()
    loss = cumulative_loss / overall 
    acc = cumulative_accuracy / overall
    clip.model.convert_weights(model)


    # print(li)
    # print(lt)
    # print(ground_truth)
    # img_loss = cost(li, ground_truth)
    # txt_loss = cost(lt, ground_truth)
    # print(img_loss)
    # print(txt_loss)

    # loss = (img_loss + txt_loss)/2
    # print(loss)
    # loss.backward()
    # optimizer.step()

    # cumulative_loss += loss.item()
    # _, predicted = li.max(dim=1)
    # cumulative_accuracy += predicted.eq(ground_truth).sum().item()

    # iou = compute_iou(bbox_gt,bbox_e)
    # running_loc_acc += iou
    # confidence = confidence[0]

    # if(iou > iou_threshold):
    #     #compare_candidate_bbox(filename,bbox_e, bbox_gt)
    #     correct_bboxes += 1
    #     overall += 1
    #     running_ga = correct_bboxes / overall
    # else:
    #     overall += 1
    #     running_ga = correct_bboxes / overall

    # # semantic = confidence[0] / 100
    # sem += confidence
    # semantic_similarity = sem / overall
    # loc_acc = running_loc_acc / overall
    # loss_meter.update(iou, confidence, filename,bbox_e,bbox_gt)
    #loss_meter.print_iteration()

    loop.set_description(f"Loss Iter = {loss}, Accuracy = {acc}")
    
loss_meter.writer.close()

  0%|          | 0/20 [00:00<?, ?it/s]

RuntimeError: expected scalar type Half but found Float